# <u> **ATM Simulation** </u>

In [1]:
import pandas as pd
import os
from datetime import datetime
import random

In [2]:
# Create Required CSV Files
ACCOUNTS_FILE = "accounts.csv"
TRANSACTIONS_FILE = "transactions.csv"

# Create accounts.csv
if not os.path.exists(ACCOUNTS_FILE):

    accounts_df = pd.DataFrame(columns=[
        "Account Number",
        "Name",
        "PIN",
        "Balance"
    ])

    accounts_df.to_csv(ACCOUNTS_FILE, index=False)

# Create transactions.csv
if not os.path.exists(TRANSACTIONS_FILE):

    transactions_df = pd.DataFrame(columns=[
        "Transaction ID",
        "Date",
        "Time",
        "Account Number",
        "Transaction Type",
        "Amount",
        "Balance"
    ])

    transactions_df.to_csv(TRANSACTIONS_FILE, index=False)

print("Required files are ready.")

Required files are ready.


In [3]:
# Global Variables
MINIMUM_BALANCE = 500

BANK_NAME = "Python National Bank"

WELCOME_LINE = "=" * 50

In [4]:
# Account Class
class Account:

    def __init__(self):
        self.accounts_file = ACCOUNTS_FILE

    # Load Account Records
    def load_accounts(self):

        return pd.read_csv(
            self.accounts_file,
            dtype={
                "Account Number": str,
                "PIN": str
            }
        )

    # Save Account Records
    def save_accounts(self, dataframe):

        dataframe.to_csv(
            self.accounts_file,
            index=False
        )

    # Generate New Account Number
    def generate_account_number(self):

        accounts = self.load_accounts()

        if accounts.empty:
            return "100001"

        last_account = accounts["Account Number"].astype(int).max()

        return str(last_account + 1)

    # Create New Account
    def create_account(self):

        accounts = self.load_accounts()

        print("\n" + "=" * 50)
        print("          CREATE NEW ACCOUNT")
        print("=" * 50)

        name = input("Enter Account Holder Name : ").strip()

        while True:

            pin = input("Create 4-digit PIN : ")

            if len(pin) != 4 or not pin.isdigit():

                print("PIN must contain exactly 4 digits.")
                continue

            confirm_pin = input("Confirm PIN : ")

            if pin != confirm_pin:

                print("PINs do not match.")
                continue

            break

        while True:

            try:

                balance = float(
                    input(
                        f"Initial Deposit (Minimum ₹{MINIMUM_BALANCE}) : "
                    )
                )

                if balance < MINIMUM_BALANCE:

                    print(
                        f"Minimum balance should be ₹{MINIMUM_BALANCE}"
                    )

                    continue

                break

            except ValueError:

                print("Please enter a valid amount.")

        account_number = self.generate_account_number()

        new_account = pd.DataFrame([{
            "Account Number": account_number,
            "Name": name,
            "PIN": pin,
            "Balance": balance
        }])

        accounts = pd.concat(
            [accounts, new_account],
            ignore_index=True
        )

        self.save_accounts(accounts)

        print("\n" + "=" * 50)
        print("Account Created Successfully!")
        print(f"Account Number : {account_number}")
        print(f"Account Holder : {name}")
        print(f"Opening Balance: ₹{balance:.2f}")
        print("=" * 50)

    # Login
    def login(self):

        accounts = self.load_accounts()

        print("\n" + "=" * 50)
        print("               LOGIN")
        print("=" * 50)

        account_number = input("Enter Account Number : ").strip()

        pin = input("Enter PIN : ").strip()

        user = accounts[
            (accounts["Account Number"] == account_number)
            &
            (accounts["PIN"] == pin)
        ]

        if user.empty:

            print("\nInvalid Account Number or PIN.")

            return None

        print("\nLogin Successful!")
        print(f"Welcome, {user.iloc[0]['Name']}!")

        return account_number

In [5]:
# Transaction Class
class Transaction:

    def __init__(self):

        self.transaction_file = TRANSACTIONS_FILE

    # Load Transactions
    def load_transactions(self):

        return pd.read_csv(
            self.transaction_file,
            dtype={
                "Transaction ID": str,
                "Account Number": str
            }
        )

    # Save Transactions
    def save_transactions(self, dataframe):

        dataframe.to_csv(
            self.transaction_file,
            index=False
        )

    # Generate Transaction ID
    def generate_transaction_id(self):

        transactions = self.load_transactions()

        if transactions.empty:
            return "TXN100001"

        last_transaction = transactions.iloc[-1]["Transaction ID"]

        transaction_number = int(
            last_transaction.replace("TXN", "")
        )

        return f"TXN{transaction_number + 1}"

    # Record a Transaction
    def log_transaction(
        self,
        account_number,
        transaction_type,
        amount,
        balance
    ):

        transactions = self.load_transactions()

        now = datetime.now()

        date = now.strftime("%d-%m-%Y")

        time = now.strftime("%I:%M:%S %p")

        transaction_id = self.generate_transaction_id()

        new_transaction = pd.DataFrame([{

            "Transaction ID": transaction_id,

            "Date": date,

            "Time": time,

            "Account Number": account_number,

            "Transaction Type": transaction_type,

            "Amount": amount,

            "Balance": balance

        }])

        transactions = pd.concat(
            [transactions, new_transaction],
            ignore_index=True
        )

        self.save_transactions(transactions)

    # Display Mini Statement
    def mini_statement(self, account_number):

        transactions = self.load_transactions()

        history = transactions[
            transactions["Account Number"] == account_number
        ]

        if history.empty:

            print("\nNo transactions found.")

            return

        print("\n" + "=" * 75)
        print("                MINI STATEMENT")
        print("=" * 75)

        print(history.tail(5).to_string(index=False))

        print("=" * 75)

    # Display Full Statement
    def full_statement(self, account_number):

        transactions = self.load_transactions()

        history = transactions[
            transactions["Account Number"] == account_number
        ]

        if history.empty:

            print("\nNo transactions found.")

            return

        print("\n" + "=" * 75)
        print("              TRANSACTION HISTORY")
        print("=" * 75)

        print(history.to_string(index=False))

        print("=" * 75)

In [6]:
# ATM Class
class ATM:

    def __init__(self):

        # Create objects of other classes
        self.account = Account()
        self.transaction = Transaction()

    # Check Balance
    def check_balance(self, account_number):

        accounts = self.account.load_accounts()

        user = accounts[
            accounts["Account Number"] == account_number
        ]

        balance = float(user.iloc[0]["Balance"])

        print("\n" + "=" * 50)
        print("          ACCOUNT BALANCE")
        print("=" * 50)
        print(f"Available Balance : ₹{balance:.2f}")
        print("=" * 50)

    # Deposit Money
    def deposit(self, account_number):

        accounts = self.account.load_accounts()

        user_index = accounts[
            accounts["Account Number"] == account_number
        ].index[0]

        while True:

            try:

                amount = float(
                    input("Enter amount to deposit : ₹")
                )

                if amount <= 0:

                    print("Amount must be greater than zero.")
                    continue

                break

            except ValueError:

                print("Please enter a valid amount.")

        current_balance = float(
            accounts.at[user_index, "Balance"]
        )

        new_balance = current_balance + amount

        accounts.at[user_index, "Balance"] = new_balance

        self.account.save_accounts(accounts)

        self.transaction.log_transaction(
            account_number,
            "Deposit",
            amount,
            new_balance
        )

        print("\n" + "=" * 50)
        print("     DEPOSIT SUCCESSFUL")
        print("=" * 50)
        print(f"Deposited Amount : ₹{amount:.2f}")
        print(f"Current Balance  : ₹{new_balance:.2f}")
        print("=" * 50)
        
    # Withdraw Money
    def withdraw(self, account_number):

        accounts = self.account.load_accounts()

        user_index = accounts[
            accounts["Account Number"] == account_number
        ].index[0]

        current_balance = float(
            accounts.at[user_index, "Balance"]
        )

        while True:

            try:

                amount = float(
                    input("Enter amount to withdraw : ₹")
                )

                if amount <= 0:

                    print("Amount must be greater than zero.")
                    continue

                if amount > current_balance:

                    print("Insufficient balance.")
                    continue

                if current_balance - amount < MINIMUM_BALANCE:

                    print(
                        f"Minimum balance of ₹{MINIMUM_BALANCE} must be maintained."
                    )
                    continue

                break

            except ValueError:

                print("Please enter a valid amount.")

        new_balance = current_balance - amount

        accounts.at[user_index, "Balance"] = new_balance

        self.account.save_accounts(accounts)

        self.transaction.log_transaction(
            account_number,
            "Withdrawal",
            amount,
            new_balance
        )

        print("\n" + "=" * 50)
        print("      WITHDRAWAL SUCCESSFUL")
        print("=" * 50)
        print(f"Withdrawn Amount : ₹{amount:.2f}")
        print(f"Current Balance  : ₹{new_balance:.2f}")
        print("=" * 50)

    # Transfer Money
    def transfer_money(self, account_number):

        accounts = self.account.load_accounts()

        sender_index = accounts[
            accounts["Account Number"] == account_number
        ].index[0]

        sender_balance = float(
            accounts.at[sender_index, "Balance"]
        )

        receiver_account = input(
            "Enter Receiver Account Number : "
        ).strip()

        if receiver_account == account_number:

            print("You cannot transfer money to your own account.")
            return

        receiver = accounts[
            accounts["Account Number"] == receiver_account
        ]

        if receiver.empty:

            print("Receiver account does not exist.")
            return

        while True:

            try:

                amount_input = input(
                    "Enter amount to transfer (or 'q' to cancel): ₹"
                )
                
                if amount_input.lower() == "q":
                    print("Transfer cancelled.")
                    return
                
                amount = float(amount_input)

                if amount <= 0:

                    print("Amount must be greater than zero.")
                    continue

                if amount > sender_balance:

                    print("Insufficient balance.")
                    continue

                if sender_balance - amount < MINIMUM_BALANCE:

                    print(
                        f"Minimum balance of ₹{MINIMUM_BALANCE} must be maintained."
                    )
                    continue

                break

            except ValueError:

                print("Please enter a valid amount.")

        receiver_index = receiver.index[0]

        sender_balance -= amount

        receiver_balance = float(
            accounts.at[receiver_index, "Balance"]
        ) + amount

        accounts.at[sender_index, "Balance"] = sender_balance

        accounts.at[receiver_index, "Balance"] = receiver_balance

        self.account.save_accounts(accounts)

        self.transaction.log_transaction(
            account_number,
            "Transfer Sent",
            amount,
            sender_balance
        )

        self.transaction.log_transaction(
            receiver_account,
            "Transfer Received",
            amount,
            receiver_balance
        )

        print("\n" + "=" * 50)
        print("      TRANSFER SUCCESSFUL")
        print("=" * 50)
        print(f"Transferred To : {receiver_account}")
        print(f"Amount         : ₹{amount:.2f}")
        print(f"Balance        : ₹{sender_balance:.2f}")
        print("=" * 50)

    # Change PIN
    def change_pin(self, account_number):

        accounts = self.account.load_accounts()

        user_index = accounts[
            accounts["Account Number"] == account_number
        ].index[0]

        current_pin = input("Enter Current PIN : ")

        if current_pin != str(accounts.at[user_index, "PIN"]):

            print("Incorrect PIN.")
            return

        while True:

            new_pin = input("Enter New 4-digit PIN : ")

            if len(new_pin) != 4 or not new_pin.isdigit():

                print("PIN must contain exactly 4 digits.")
                continue

            confirm_pin = input("Confirm New PIN : ")

            if new_pin != confirm_pin:

                print("PINs do not match.")
                continue

            break

        accounts.at[user_index, "PIN"] = new_pin

        self.account.save_accounts(accounts)

        print("\nPIN changed successfully.")
    # User Menu
    def user_menu(self, account_number):

        while True:

            print("\n" + "=" * 50)
            print(f"      Welcome to {BANK_NAME}")
            print("=" * 50)
            print("1. Check Balance")
            print("2. Deposit Money")
            print("3. Withdraw Money")
            print("4. Transfer Money")
            print("5. Mini Statement")
            print("6. Full Statement")
            print("7. Change PIN")
            print("8. Logout")
            print("=" * 50)

            choice = input("Enter your choice : ")

            if choice == "1":

                self.check_balance(account_number)

            elif choice == "2":

                self.deposit(account_number)

            elif choice == "3":

                self.withdraw(account_number)

            elif choice == "4":

                self.transfer_money(account_number)

            elif choice == "5":

                self.transaction.mini_statement(account_number)

            elif choice == "6":

                self.transaction.full_statement(account_number)

            elif choice == "7":

                self.change_pin(account_number)

            elif choice == "8":

                print("\nLogged out successfully.")

                break

            else:

                print("Invalid choice. Please try again.")

    # Main Menu
    def main_menu(self):

        while True:

            print("\n" + "=" * 50)
            print(f"        {BANK_NAME}")
            print("=" * 50)
            print("1. Create Account")
            print("2. Login")
            print("3. Exit")
            print("=" * 50)

            choice = input("Enter your choice : ")

            if choice == "1":

                self.account.create_account()

            elif choice == "2":

                account_number = self.account.login()

                if account_number is not None:

                    self.user_menu(account_number)

            elif choice == "3":

                print("\nThank you for using our ATM.")
                print("Have a nice day!")

                break

            else:

                print("Invalid choice. Please try again.")

In [7]:
atm = ATM()
atm.main_menu()


        Python National Bank
1. Create Account
2. Login
3. Exit


Enter your choice :  2



               LOGIN


Enter Account Number :  100002
Enter PIN :  8062



Login Successful!
Welcome, John Markson!

      Welcome to Python National Bank
1. Check Balance
2. Deposit Money
3. Withdraw Money
4. Transfer Money
5. Mini Statement
6. Full Statement
7. Change PIN
8. Logout


...


              TRANSACTION HISTORY
Transaction ID       Date        Time Account Number  Transaction Type  Amount  Balance
     TXN100001 09-08-2026 09:26:45 AM         100001           Deposit   500.0   1000.0
     TXN100002 09-08-2026 09:27:00 AM         100001        Withdrawal   250.0    750.0
     TXN100005 09-08-2026 09:36:51 AM         100001 Transfer Received   500.0   1250.0

      Welcome to Python National Bank
1. Check Balance
2. Deposit Money
3. Withdraw Money
4. Transfer Money
5. Mini Statement
6. Full Statement
7. Change PIN
8. Logout


Enter your choice :  8



Logged out successfully.

        Python National Bank
1. Create Account
2. Login
3. Exit


Enter your choice :  3



Thank you for using our ATM.
Have a nice day!
